Vamos a tantear diferentes modelos y configuraciones antes de parametrizar el definitivo. Gracias a la libreria lazypredict hara un tanteo en los principales modelos de regresion y clasificacion.



In [8]:
import pandas as pd
import numpy as np, random
random.seed(42)

In [9]:
df2 = pd.read_csv('C:/Users/Josue/4GA.Datascience/4GA.DataScience/models/presplit.csv')

In [10]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12800 entries, 0 to 12799
Data columns (total 34 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   cntry                           12800 non-null  object 
 1   pplfair                         12800 non-null  int64  
 2   pplhlp                          12800 non-null  int64  
 3   ppltrst                         12800 non-null  int64  
 4   lrscale                         12800 non-null  int64  
 5   polintr                         12800 non-null  int64  
 6   stfdem                          12800 non-null  int64  
 7   stfeco                          12800 non-null  int64  
 8   stfgov                          12800 non-null  int64  
 9   trstep                          12800 non-null  int64  
 10  trstlgl                         12800 non-null  int64  
 11  trstplc                         12800 non-null  int64  
 12  trstplt                         

In [11]:
import pandas as pd

def redondear_columnas(df, columnas):

    for columna in columnas:
        if columna in df.columns:
            df[columna] = df[columna].apply(lambda x: round(x, 1) if pd.notnull(x) else x)
        else:
            print(f"La columna '{columna}' no existe en el DataFrame.")
    return df

columnas_a_redondear = ['confianza_promedio', 'satisf_media', 'ppl']

# Aplicar el redondeo a las columnas especificadas
df2= redondear_columnas(df2, columnas_a_redondear)

# Verificar los valores redondeados
print(df2.head(4))

          cntry  pplfair  pplhlp  ppltrst  lrscale  polintr  stfdem  stfeco  \
0        España        5       2        8        8        2       9       8   
1      Portugal        5       5        5        0        4       5       6   
2  Países Bajos        6       8        6        6        3       6       8   
3      Alemania        7       7       10        5        3       3       5   

   stfgov  trstep  ...  happyfc  confianza_promedio  \
0       8       8  ...        2                 5.0   
1       5       5  ...        1                 4.7   
2       7       4  ...        1                 4.9   
3       2       2  ...        2                 3.6   

   confianza_promedio_factorizada  satisf_media  satisf_media_factorizada  \
0                               2           8.3                         4   
1                               2           5.3                         2   
2                               2           7.0                         3   
3                   

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
import os


def bestclass(df, target_column, feature_range, exclude_columns=None):


    results = {}
    X = df.drop(target_column, axis=1)
    y = df[target_column]

    if exclude_columns:
        X = X.drop(exclude_columns, axis=1)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    classifiers = {
        "Random Forest": RandomForestClassifier(random_state=42),
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        "LightGBM": LGBMClassifier(random_state=42),
        "SVM": SVC(probability=True, random_state=42)
    }

    best_model = None
    best_auc = -float('inf')
    best_k = 0
    best_features = []
    best_classifier_name = ""

    for k in range(feature_range[0], feature_range[1] + 1):
        for classifier_name, classifier in classifiers.items():
            selector = SelectKBest(score_func=f_classif, k=k)
            X_train_selected = selector.fit_transform(X_train, y_train)
            X_test_selected = selector.transform(X_test)

            classifier.fit(X_train_selected, y_train)
            y_pred_prob = classifier.predict_proba(X_test_selected)

            auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr')

            if auc > best_auc:
                best_auc = auc
                best_model = classifier
                best_k = k
                best_features = X.columns[selector.get_support()].tolist()
                best_classifier_name = classifier_name

    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/App'
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, 'bestclassifier_iterative_corrected.txt'), 'w') as f:
        f.write(f"Mejor modelo: {best_classifier_name}\n")
        f.write(f"AUC: {best_auc}\n")
        f.write(f"Número de características: {best_k}\n")
        f.write(f"Características: {best_features}\n")

    return best_model, best_k, best_features, best_classifier_name


exclude_cols = ['lrscale', 'lrscale2_fc','cntry','lw_pnd']  
best_model, best_k, best_features, best_classifier_name = bestclass(df2, 'lrscale_fc', (10, 18), exclude_columns=exclude_cols)

print(f"Mejor modelo: {best_classifier_name}")
print(f"Número de características: {best_k}")
print(f"Características: {best_features}")

c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\xgboost\training.py:183: UserWarning: [06:19:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0,000105 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 98
[LightGBM] [Info] Number of data points in the train set: 10240, number of used features: 10
[LightGBM] [Info] Start training from score -1,971428
[LightGBM] [Info] Start training from score -1,465524
[LightGBM] [Info] Start training from score -0,665031
[LightGBM] [Info] Start training from score -2,158248


c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\xgboost\training.py:183: UserWarning: [06:19:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0,000073 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 109
[LightGBM] [Info] Number of data points in the train set: 10240, number of used features: 11
[LightGBM] [Info] Start training from score -1,971428
[LightGBM] [Info] Start training from score -1,465524
[LightGBM] [Info] Start training from score -0,665031
[LightGBM] [Info] Start training from score -2,158248


c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\xgboost\training.py:183: UserWarning: [06:19:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0,000093 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 227
[LightGBM] [Info] Number of data points in the train set: 10240, number of used features: 17
[LightGBM] [Info] Start training from score -1,971428
[LightGBM] [Info] Start training from score -1,465524
[LightGBM] [Info] Start training from score -0,665031
[LightGBM] [Info] Start training from score -2,158248


c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\xgboost\training.py:183: UserWarning: [06:22:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0,000098 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 238
[LightGBM] [Info] Number of data points in the train set: 10240, number of used features: 18
[LightGBM] [Info] Start training from score -1,971428
[LightGBM] [Info] Start training from score -1,465524
[LightGBM] [Info] Start training from score -0,665031
[LightGBM] [Info] Start training from score -2,158248


c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Mejor modelo: SVM
Número de características: 17
Características: ['polintr', 'stfdem', 'stfeco', 'stfgov', 'trstep', 'trstlgl', 'trstprl', 'imbgeco', 'imwbcnt', 'rlgdgr', 'cntgrp_fc', 'cnt_fc', 'rel3fc', 'confianza_promedio', 'satisf_media', 'satisf_media_factorizada', 'pintfc']


Es increible que ha dado un 0.67 de score con SVM , cuando svm clasifica casi obligatoriamente con datos escalados.
Vamos a escalar las variables numericas no factorizadas.
E hiperparametrizamos.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import os
#probamos con la variante lrscale de 5 opciones factorizadas
def best_svm_scaled(df, target_column, feature_range, scale_columns=None, exclude_columns=None):


    results = {}
    X = df.drop(target_column, axis=1)
    y = df[target_column]

    if exclude_columns:
        X = X.drop(exclude_columns, axis=1)

    if scale_columns:
        
        scaler_minmax = MinMaxScaler()
        X_minmax = X.copy()
        X_minmax[scale_columns] = scaler_minmax.fit_transform(X_minmax[scale_columns])

        
        scaler_std = StandardScaler()
        X_std = X.copy()
        X_std[scale_columns] = scaler_std.fit_transform(X_std[scale_columns])

       
        X_combined = pd.concat([X, X_minmax[scale_columns].add_suffix('_minmax'), X_std[scale_columns].add_suffix('_std')], axis=1)
    else:
        X_combined = X.copy()

    X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

    best_model = None
    best_auc = -float('inf')
    best_k = 0
    best_features = []
    best_accuracy = 0
    best_precision = 0
    best_recall = 0
    best_f1 = 0

    for k in range(feature_range[0], feature_range[1] + 1):
        selector = SelectKBest(score_func=f_classif, k=k)
        X_train_selected = selector.fit_transform(X_train, y_train)
        X_test_selected = selector.transform(X_test)

        classifier = SVC(probability=True, random_state=42)
        classifier.fit(X_train_selected, y_train)
        y_pred_prob = classifier.predict_proba(X_test_selected)
        y_pred = classifier.predict(X_test_selected)

        auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr')
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        if auc > best_auc:
            best_auc = auc
            best_model = classifier
            best_k = k
            best_features = X_combined.columns[selector.get_support()].tolist()
            best_accuracy = accuracy
            best_precision = precision
            best_recall = recall
            best_f1 = f1

    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/App'
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, 'BestSVMdeflr4_scaled.txt'), 'w') as f:
        f.write(f"Mejor modelo: SVM\n")
        f.write(f"AUC: {best_auc}\n")
        f.write(f"Accuracy: {best_accuracy}\n")
        f.write(f"Precision: {best_precision}\n")
        f.write(f"Recall: {best_recall}\n")
        f.write(f"F1-score: {best_f1}\n")
        f.write(f"Número de características: {best_k}\n")
        f.write(f"Características: {best_features}\n")

    return best_model, best_k, best_features, best_accuracy, best_precision, best_recall, best_f1


exclude_cols = ['lrscale', 'lrscale2_fc', 'cntry', 'lw_pnd']
scale_cols = ['confianza_promedio', 'satisf_media', 'ppl']

best_model, best_k, best_features, best_accuracy, best_precision, best_recall, best_f1 = best_svm_scaled(df2, 'lrscale_fc', (10, 20), scale_columns=scale_cols, exclude_columns=exclude_cols)

print(f"Número de características: {best_k}")
print(f"Características: {best_features}")
print(f"Accuracy: {best_accuracy}")
print(f"Precision: {best_precision}")
print(f"Recall: {best_recall}")
print(f"F1-score: {best_f1}")

c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(r

Número de características: 20
Características: ['polintr', 'stfdem', 'stfeco', 'stfgov', 'trstlgl', 'trstprl', 'imbgeco', 'imwbcnt', 'rlgdgr', 'cntgrp_fc', 'cnt_fc', 'rel3fc', 'confianza_promedio', 'satisf_media', 'satisf_media_factorizada', 'pintfc', 'confianza_promedio_minmax', 'satisf_media_minmax', 'confianza_promedio_std', 'satisf_media_std']
Accuracy: 0.541015625
Precision: 0.5470936178432011
Recall: 0.541015625
F1-score: 0.39897590913988956


In [18]:
#probamos con la variante lrscale de 5 opciones factorizadas
exclude_cols = ['lrscale', 'lrscale_fc', 'cntry', 'lw_pnd']
scale_cols = ['confianza_promedio', 'satisf_media', 'ppl']
best_model, best_k, best_features, best_accuracy, best_precision, best_recall, best_f1 = best_svm_scaled(df2, 'lrscale2_fc', (10, 20), scale_columns=scale_cols, exclude_columns=exclude_cols)
print(f"Número de características: {best_k}")
print(f"Características: {best_features}")
print(f"Accuracy: {best_accuracy}")
print(f"Precision: {best_precision}")
print(f"Recall: {best_recall}")
print(f"F1-score: {best_f1}")

c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(r

Número de características: 19
Características: ['polintr', 'stfdem', 'stfeco', 'stfgov', 'trstplt', 'imbgeco', 'imwbcnt', 'rlgdgr', 'cntgrp_fc', 'cnt_fc', 'rel3fc', 'confianza_promedio', 'satisf_media', 'satisf_media_factorizada', 'pintfc', 'confianza_promedio_minmax', 'satisf_media_minmax', 'confianza_promedio_std', 'satisf_media_std']
Accuracy: 0.440625
Precision: 0.3886391129032258
Recall: 0.440625
F1-score: 0.28613912231075855


c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import os
#Probamos con criterio chi2


def best_svm_chi2_minmax(df, target_column, feature_range, scale_columns=None, exclude_columns=None, transform_negative=False):


    results = {}
    X = df.drop(target_column, axis=1)
    y = df[target_column]

    if exclude_columns:
        X = X.drop(exclude_columns, axis=1)

    # Inicializar X_combined antes del condicional
    X_combined = X.copy()

    if scale_columns:
        # Generar variables escaladas con MinMaxScaler
        scaler_minmax = MinMaxScaler()
        X_minmax = X.copy()
        X_minmax[scale_columns] = scaler_minmax.fit_transform(X_minmax[scale_columns])

        # Concatenar las variables originales y escaladas
        X_combined = pd.concat([X, X_minmax[scale_columns].add_suffix('_minmax')], axis=1)

    X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

    best_model = None
    best_auc = -float('inf')
    best_k = 0
    best_features = []
    best_accuracy = 0
    best_precision = 0
    best_recall = 0
    best_f1 = 0

    for k in range(feature_range[0], feature_range[1] + 1):
        selector = SelectKBest(score_func=chi2, k=k)

        if transform_negative:
            X_train_selected = selector.fit_transform(X_train - X_train.min(), y_train)
            X_test_selected = selector.transform(X_test - X_test.min())
        else:
            X_train_selected = selector.fit_transform(X_train, y_train)
            X_test_selected = selector.transform(X_test)

        classifier = SVC(probability=True, random_state=42)
        classifier.fit(X_train_selected, y_train)
        y_pred_prob = classifier.predict_proba(X_test_selected)
        y_pred = classifier.predict(X_test_selected)

        auc = roc_auc_score(y_test, y_pred_prob, multi_class='ovr')
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        if auc > best_auc:
            best_auc = auc
            best_model = classifier
            best_k = k
            best_features = X_combined.columns[selector.get_support()].tolist()
            best_accuracy = accuracy
            best_precision = precision
            best_recall = recall
            best_f1 = f1

    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/App'
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, 'BestSVMdeflr4_chi2_minmax.txt'), 'w') as f:
        f.write(f"Mejor modelo: SVM\n")
        f.write(f"AUC: {best_auc}\n")
        f.write(f"Accuracy: {best_accuracy}\n")
        f.write(f"Precision: {best_precision}\n")
        f.write(f"Recall: {best_recall}\n")
        f.write(f"F1-score: {best_f1}\n")
        f.write(f"Número de características: {best_k}\n")
        f.write(f"Características: {best_features}\n")

    return best_model, best_k, best_features, best_accuracy, best_precision, best_recall, best_f1


exclude_cols = ['lrscale', 'lrscale2_fc', 'cntry', 'lw_pnd']
scale_cols = ['confianza_promedio', 'satisf_media', 'ppl']

best_model, best_k, best_features, best_accuracy, best_precision, best_recall, best_f1 = best_svm_chi2_minmax(df2, 'lrscale_fc', (7, 20), scale_columns=scale_cols, exclude_columns=exclude_cols, transform_negative=True)

print(f"Número de características: {best_k}")
print(f"Características: {best_features}")
print(f"Accuracy: {best_accuracy}")
print(f"Precision: {best_precision}")
print(f"Recall: {best_recall}")
print(f"F1-score: {best_f1}")

c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(r

Número de características: 16
Características: ['ppltrst', 'polintr', 'stfdem', 'stfeco', 'stfgov', 'trstep', 'trstlgl', 'trstplt', 'trstprl', 'imbgeco', 'imwbcnt', 'rlgdgr', 'cntgrp_fc', 'cnt_fc', 'rel3fc', 'satisf_media']
Accuracy: 0.544921875
Precision: 0.559275503676969
Recall: 0.544921875
F1-score: 0.411087474810554


In [26]:
exclude_cols = ['lrscale', 'lrscale_fc', 'cntry', 'lw_pnd']
scale_cols = ['confianza_promedio', 'satisf_media', 'ppl']
best_model, best_k, best_features, best_accuracy, best_precision, best_recall, best_f1 = best_svm_chi2_minmax(df2, 'lrscale2_fc', (7, 20), scale_columns=scale_cols, exclude_columns=exclude_cols, transform_negative=True)
print(f"Número de características: {best_k}")
print(f"Características: {best_features}")
print(f"Accuracy: {best_accuracy}")
print(f"Precision: {best_precision}")
print(f"Recall: {best_recall}")
print(f"F1-score: {best_f1}")

c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Josue\4GA.Datascience\.venv3129\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(r

Número de características: 18
Características: ['ppltrst', 'polintr', 'stfdem', 'stfeco', 'stfgov', 'trstlgl', 'trstplt', 'trstprl', 'trstprt', 'imbgeco', 'imwbcnt', 'rlgdgr', 'cntgrp_fc', 'cnt_fc', 'rel3fc', 'confianza_promedio', 'satisf_media', 'satisf_media_factorizada']
Accuracy: 0.442578125
Precision: 0.3487712030825948
Recall: 0.442578125
F1-score: 0.29473438402261687


In [33]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import os

def best_ordinal_regression(df, target_column, scale_columns=None, exclude_columns=None):
 


    results = {}
    X = df.drop(target_column, axis=1)
    y = df[target_column]

    if exclude_columns:
        X = X.drop(exclude_columns, axis=1)

    # Inicializar X_scaled
    X_scaled = X.copy()

    if scale_columns:
        # Generar variables escaladas con MinMaxScaler
        scaler_minmax = MinMaxScaler()
        X_scaled[scale_columns] = scaler_minmax.fit_transform(X[scale_columns])

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

    # Añadir constante para statsmodels
    X_train = sm.add_constant(X_train)
    X_test = sm.add_constant(X_test)

    # Modelo de regresión ordinal
    model = OrderedModel(y_train, X_train, dist='logit')  # 'logit' para regresión logística ordinal
    result = model.fit(method='bfgs')

    # Predicciones
    predictions = result.predict(X_test)

    # Obtener la clase predicha con la mayor probabilidad
    y_pred = predictions.apply(lambda x: x.idxmax(), axis=1)

    # Guardar resultados
    output_dir = 'C:/Users/Josue/4GA.Datascience/4GA.DataScience/App'
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, 'OrdinalRegressionResults.txt'), 'w') as f:
        f.write(f"Resultados de la Regresión Ordinal:\n")
        f.write(f"{result.summary()}\n")

    return result, y_pred, y_test

# Ejemplo de uso (asumiendo que df2 está definido):
exclude_cols = ['lrscale_fc', 'lrscale2_fc', 'cntry', 'lw_pnd']
scale_cols = ['confianza_promedio', 'satisf_media', 'ppl']

model_results, y_pred, y_test = best_ordinal_regression(df2, 'lrscale', scale_columns=scale_cols, exclude_columns=exclude_cols)

print(model_results.summary())
print(f"Predicciones:\n{y_pred}")
print(f"Valores reales:\n{y_test}")

ModuleNotFoundError: No module named 'statsmodels'